# Computer Vision

Let's do some very basic computer vision. We're going to import the MNIST handwritten digits data and $k$NN to predict values (i.e. "see/read").

1. To load the data, run the following code in a chunk:
```
import pickle
with open('data/minst.pkl', 'rb') as f:
    data = pickle.load(f)
X_train, y_train = data['X_train'], data['y_train']
X_test, y_test = data['X_test'], data['y_test']
```
The `y_test` and `y_train` vectors, for each index `i`, tell you want number is written in the corresponding index in `X_train[i]` and `X_test[i]`. The value of `X_train[i]` and `X_test[i]`, however, is a 28$\times$28 array whose entries contain values between 0 and 256. Each element of the matrix is essentially a "pixel" and the matrix encodes a representation of a number. To visualize this, run the following code to see the first ten numbers:
```
import matplotlib.pyplot as plt
import numpy as np
np.set_printoptions(edgeitems=30, linewidth=100000)
for i in range(5): 
    print(y_test[i],'\n') # Print the label
    print(X_test[i],'\n') # Print the matrix of values
    plt.contourf(np.rot90(X_test[i].transpose())) # Make a contour plot of the matrix values
    plt.show()
```
OK, those are the data: Labels attached to handwritten digits encoded as a matrix.

2. What is the shape of `X_train` and `X_test`? What is the shape of `X_train[i]` and `X_test[i]` for each index `i`? What is the shape of `y_train` and `y_test`?
3. Use Numpy's `.reshape()` method to covert the training and testing data from a matrix into an vector of features. So, `X_test[index].reshape((1,784))` will convert the $index$-th element of `X_test` into a $28\times 28=784$-length row vector of values, rather than a matrix. Turn `X_train` into an $N \times 784$ matrix $X$ that is suitable for scikit-learn's kNN classifier where $N$ is the number of observations and $784=28*28$ (you could use, for example, a `for` loop).
4. Use the reshaped `X_test` and `y_test` data to create a $k$-nearest neighbor classifier of digit. What is the optimal number of neighbors $k$? If you can't determine this, play around with different values of $k$ for your classifier.
5. For the optimal number of neighbors, how well does your predictor perform on the test set? Report the accuracy, compute a confusion matrix, and explain your findings.
6. For your confusion matrix, which mistakes are most likely? Do you find any interesting patterns?
7. So, this is how computers "see." They convert an image into a matrix of values, that matrix becomes a vector in a dataset, and then we deploy ML tools on it as if it was any other kind of tabular data. To make sure you follow this, invent a way to represent a color photo in matrix form, and then describe how you could convert it into tabular data. (Hint: RGB color codes provide a method of encoding a numeric value that represents a color.)

---

## *Phase 1: Solution without AI

Complete all questions in the following cells without generative AI. Then commit and push the
notebook before beginning Phase 2.

**Declaration:** I completed this version without generative AI.

**Student(s):** [To be provided]

**Date:** [To be provided]

In [4]:
# Your Phase 1 solution to the problem goes here.
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
import numpy as np


import pickle
with open('data/minst.pkl', 'rb') as f:
    data = pickle.load(f)
X_train, y_train = data['X_train'], data['y_train']
X_test, y_test = data['X_test'], data['y_test']
# reshaping 
# Q3: Reshape 28x28 images into vectors of 784 features

X_train_flat = X_train.reshape(X_train.shape[0], 784)
X_test_flat = X_test.reshape(X_test.shape[0], 784)

print("Q3: Reshaped data")
print("Training:", X_train_flat.shape)
print("Testing:", X_test_flat.shape)

assert X_train_flat.shape[1] == 784
assert X_test_flat.shape[1] == 784
assert X_train_flat.shape[0] == len(y_train)
assert X_test_flat.shape[0] == len(y_test)

print("All checks passed.")



# q4 creating knn from reshaped data

print("Building k-NN Classifier\n")
evals_ks = np.array([1, 3, 5, 7, 9])
eval_valid_accuracy = []

# iterate through each K
for eval_k in evals_ks:
    eval_candidate = KNeighborsClassifier(n_neighbors=int(eval_k))
    eval_candidate.fit(X_train_flat, y_train)
    eval_valid_hat = eval_candidate.predict(X_test_flat)
    eval_valid_accuracy.append(
        accuracy_score(y_test, eval_valid_hat)
    )
    print("k =", eval_k, "accuracy =", accuracy_score(y_test, eval_valid_hat))

eval_k_star = int(evals_ks[np.argmax(eval_valid_accuracy)])
print("\nhighest k value: ", eval_k_star)

Q3: Reshaped data
Training: (60000, 784)
Testing: (10000, 784)
All checks passed.
Building k-NN Classifier



KeyboardInterrupt: 

In [ ]:

# Q5.  computing confusion matrix

# train classification model with optimal k
classification_model = KNeighborsClassifier(n_neighbors=eval_k_star).fit(
    X_train_flat, y_train
)

y_valid_predict = classification_model.predict(X_test_flat)

# confusion matrix
print("confusion matrix\n")
cm = confusion_matrix(y_test, y_valid_predict)
print(cm)

# check accuracy
accuracy = accuracy_score(y_test, y_valid_predict)
print("Accuracy: ", accuracy)

confusion matrix

[[ 974    1    1    0    0    1    2    1    0    0]
 [   0 1133    2    0    0    0    0    0    0    0]
 [  10    9  996    2    0    0    0   13    2    0]
 [   0    2    4  976    1   13    1    7    3    3]
 [   1    6    0    0  950    0    4    2    0   19]
 [   6    1    0   11    2  859    5    1    3    4]
 [   5    3    0    0    3    3  944    0    0    0]
 [   0   21    5    0    1    0    0  991    0   10]
 [   8    2    4   16    8   11    3    4  914    4]
 [   4    5    2    8    9    2    1    8    2  968]]
Accuracy:  0.9705


Question 4:
We tested odd values of k from 1 to 9 to help us avoid having ties a determine which number of neighbors gave the best accuracy.

The test accuracies were:

- k=1 : 96.91%
- k=3 : 97.05% 
- k=5: 96.88%
- k=7: 96.94%
- k=9: 96.59%

Based on these results, the optimal number of neighbors is k=3 with had an accuracy of 97.05%.

This shows the balance between overfitting and underfitting. With k= 1, the model is more sensitive to noise. At k=3, it performs better by considering nearby examples while still recognizing differences between digits. As k increases, accuracy generally decreases since the model considers more neighbors, which can make it harder to distinguish between similar-looking digits.



Question 5:

Using the optimal k-classifier, 3, achieved an overall test accuracy of 97.05%, meaning it correctly classified the majority of the handwritten digits.

Confusion Matrix Analysis:

Along the diagonal of the  matrix showed that the model correctly classified most of the digists. Digit 1 had the highest number of correct predictions (1,133), and had only 2 errors. Digit 0 also performed pretty well, with 974 correct predictions and 5 errors. Some of the common misclassifications, was 7 predicted as 1, which was the most common error that happened 21 times. 4 was predicted as 9 which happened 19 times, which is likely because depending on how 4s are written they have a close shape to 9. 8 was also commonly predicted as 3, 16 times and as 5, 11 times. Overall, the model performed very well, but the  errors show that it can still struggle with digits that have similar handwritten shapes.


## *Phase 2: AI-assisted Revision
Keep Phase 1 frozen.

- **Phase 1 commit SHA ID:** [To be provided]
- **AI tool and model used:** [To be provided]

### *Prompts used

1. [Paste prompt]
2. [Paste follow-up prompt]

### *AI-proposed changes


[Paste the AI's concise numbered list verbatim.]

1. ...
2. ...

### *Evaluation of AI suggestions


| Change | Decision | Reason and verification |
|---|---|---|
| 1 | Accept / modify / reject | ... |
| 2 | Accept / modify / reject | ... |

### *AI-assisted solution in full
After the evaluation of AI suggestions, incorporate the accepted revisions into your Phase 1 solution and provide the final Phase 2 solution in full in the following cells.

In [ ]:
# Your Phase 2 solution to the problem goes here.


### *Reflection on AI assistance
In your own words without generative AI.

[In 150–300 words, describe the main improvements, AI mistakes or limitations, how the final solution was verified, and any remaining concerns.]